# Ejemplo 4 · Búsqueda en contratos
**Estud-IA · Clase 5 · Intermedio.**

**Problema.** Tienes muchas cláusulas de contratos y quieres encontrar rápido las que tratan de un tema.

**El modelo, en palabras simples (búsqueda por similitud con TF-IDF).**
No se entrena nada. Cada cláusula y tu pregunta se convierten en números según las palabras que usan (TF-IDF: las palabras raras pesan más). Luego se mide qué cláusulas se parecen más a la pregunta (similitud del coseno) y se devuelven las mejores.

**Su límite:** compara **palabras**, no **significado**. Si preguntas «multa» y el contrato dice «penalidad», no las relaciona. Hay mejoras posibles y aquí se prueban dos: letras sueltas (n-gramas de caracteres) y, como variante a medir tú, representar el significado con un modelo de lenguaje (embeddings).

**Datos.** 24 cláusulas **ficticias**, sin datos de ninguna empresa. Con contratos reales no se suben a una IA pública: usa datos ficticios.

## Parte 0 · Las cláusulas

In [ ]:
import numpy as np, pandas as pd
clausulas = [
 "El proveedor pagará una penalidad del dos por ciento por cada día de retraso en la entrega.",
 "Si el contratista se atrasa en la entrega se aplicará una multa diaria sobre el valor del pedido.",
 "Las partes mantendrán confidencial toda la información técnica y comercial intercambiada.",
 "Ninguna parte divulgará a terceros los secretos empresariales de la otra.",
 "El cliente puede terminar el contrato antes de tiempo con aviso escrito de treinta días.",
 "El contrato podrá darse por terminado de manera anticipada si una parte incumple sus obligaciones.",
 "El pago se hará dentro de los treinta días siguientes a la recepción de la factura.",
 "Las facturas se cancelarán a los sesenta días de su radicación en la oficina del cliente.",
 "La propiedad intelectual del software desarrollado pertenece al cliente una vez pagado el valor total.",
 "Los derechos de autor sobre los entregables serán del cliente desde la fecha de entrega.",
 "Ninguna parte responde por incumplimientos causados por fuerza mayor o caso fortuito.",
 "Un terremoto, una inundación o una pandemia eximen de responsabilidad por la demora.",
 "El proveedor garantiza los equipos por doce meses contra defectos de fabricación.",
 "Durante un año el vendedor reparará sin costo cualquier falla de los productos entregados.",
 "Las diferencias se resolverán ante los jueces de la ciudad de Medellín.",
 "Cualquier conflicto será sometido a un tribunal de arbitramento de la cámara de comercio.",
 "El precio no incluye impuestos, que serán asumidos por el cliente.",
 "El valor del contrato podrá reajustarse cada año según el índice de precios al consumidor.",
 "El contratista debe mantener vigente una póliza de cumplimiento durante la ejecución.",
 "El proveedor contratará un seguro que cubra los daños causados a terceros.",
 "El personal del proveedor no tendrá vínculo laboral con el cliente.",
 "Las comunicaciones entre las partes se harán por correo electrónico a las direcciones indicadas.",
 "El cliente podrá auditar los registros del proveedor con previo aviso de diez días.",
 "Este contrato no podrá cederse a un tercero sin autorización escrita de la otra parte.",
]
tema = [0,0,1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,11,12,13]
nombres = ["retraso","confidencialidad","terminación","pagos","propiedad intelectual","fuerza mayor","garantía","jurisdicción","precio","seguros","personal","comunicaciones","auditoría","cesión"]
print(len(clausulas), "cláusulas | cada tema tiene sus parejas de redacción distinta (las 2 primeras tratan el mismo tema)")

## Parte 1 · Armar el buscador
**Qué hace:** convierte las cláusulas en números y deja listo el buscador. Cero entrenamiento.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def armar(vectorizador):
    M = vectorizador.fit_transform(clausulas)
    def buscar(pregunta, k=3):
        s = cosine_similarity(vectorizador.transform([pregunta]), M)[0]
        return np.argsort(s)[::-1][:k], np.sort(s)[::-1][:k]
    return buscar

buscar_palabras = armar(TfidfVectorizer())
ids, sc = buscar_palabras("penalidad por retraso en la entrega")
for i, s in zip(ids, sc): print(f"{s:.2f} | {clausulas[i]}")

## Parte 2 · Medir: ¿sale la cláusula correcta?
Se escriben preguntas y, **antes de ejecutar**, se anota qué cláusulas tratan el tema (cada tema tiene 2). La medida: ¿la primera cláusula devuelta es del tema correcto? Respuesta en «k de N».

Hay dos tipos de preguntas: **con las mismas palabras del contrato** y **con otras palabras** (sinónimos). **Cámbialas por las tuyas.**

In [ ]:
preguntas = [
 # mismas palabras que el contrato
 ("penalidad por retraso en la entrega", 0, "mismas palabras"),
 ("información confidencial", 1, "mismas palabras"),
 ("pago de la factura", 3, "mismas palabras"),
 ("garantía de los equipos", 6, "mismas palabras"),
 # otras palabras (sinónimos)
 ("castigo si llegan tarde los productos", 0, "otras palabras"),
 ("guardar secreto de lo que nos contamos", 1, "otras palabras"),
 ("quién es dueño del programa que hagan", 4, "otras palabras"),
 ("qué pasa si hay un desastre natural", 5, "otras palabras"),
]
def evaluar(buscar):
    filas = []
    for q, t, tipo in preguntas:
        ids, _ = buscar(q, 1); filas.append({"pregunta": q, "tipo": tipo, "tema esperado": nombres[t], "devolvió": nombres[tema[ids[0]]], "¿acierta?": tema[ids[0]] == t})
    return pd.DataFrame(filas)
r1 = evaluar(buscar_palabras); display(r1)
for tipo in ["mismas palabras", "otras palabras"]:
    s = r1[r1.tipo == tipo]; print(f"{tipo}: {int(s['¿acierta?'].sum())} de {len(s)}")

## Parte 3 · Un solo cambio: letras en vez de palabras
**Cambio:** comparar por trozos de 3 a 5 letras (n-gramas de caracteres). Ayuda con variaciones de la misma palabra («atraso», «atrasa», «atrasarse»). Todo lo demás queda igual.

In [ ]:
buscar_letras = armar(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)))
r2 = evaluar(buscar_letras)
for tipo in ["mismas palabras", "otras palabras"]:
    a = r1[r1.tipo == tipo]; b = r2[r2.tipo == tipo]
    print(f"{tipo}: palabras {int(a['¿acierta?'].sum())} de {len(a)}  ->  letras {int(b['¿acierta?'].sum())} de {len(b)}")

**Cómo leer esto.** Compara lo que cambió, no solo el total. Un trozo de 3 a 5 letras no sabe que «castigo» y «penalidad» significan lo mismo: para eso hace falta representar el **significado**.

## Parte 4 · La mejora que se mide tú: embeddings (significado)
Con un modelo de lenguaje que convierte cada frase en una lista de números según su significado (el *encoder* `paraphrase-multilingual-MiniLM-L12-v2`, que ya usamos en esta clase), «castigo si llegan tarde» queda cerca de «penalidad por retraso».

**Esta celda no la ejecuté en mi sandbox (no tiene acceso a Hugging Face): corre en Colab y mide tú el resultado con el mismo «k de N» antes de creerle.** Si no descarga, no pasa nada: el resto del notebook ya funciona.

In [ ]:
# OPCIONAL · requiere internet y descarga ~120 MB. Ejecútala en Colab.
try:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    E = enc.encode(clausulas, normalize_embeddings=True)
    def buscar_significado(q, k=3):
        s = (enc.encode([q], normalize_embeddings=True) @ E.T)[0]
        return np.argsort(s)[::-1][:k], np.sort(s)[::-1][:k]
    r3 = evaluar(buscar_significado)
    for tipo in ["mismas palabras", "otras palabras"]:
        s = r3[r3.tipo == tipo]; print(f"{tipo}: significado {int(s['¿acierta?'].sum())} de {len(s)}")
except Exception as e:
    print("No se pudo cargar el modelo de significado:", type(e).__name__, "- continúa con lo anterior.")

## Parte 5 · Tu turno: tu ficha convertida en prompt
Ficha (ejemplo): *Camino A, sin entrenar. Cambio: n-gramas de letras. Medida: k de N en 8 preguntas (4 con las palabras del contrato, 4 con sinónimos). Criterio: adopto el cambio si gana en «otras palabras» por 2 o más. Riesgo: pocas cláusulas, preguntas mal escritas.*

Prompt para tu IA:
```
Tengo cláusulas de contratos ficticias en una columna llamada clausula.
Hazme un notebook de Colab con una búsqueda por similitud de texto (TF-IDF): yo escribo una pregunta
y me muestras las 3 cláusulas más parecidas con su puntaje.
Incluye una tabla de 8 preguntas (4 con las mismas palabras y 4 con sinónimos) y cuántas acertó de cuántas.
Si no tengo cláusulas, genera 24 ficticias. Explica cada celda sin matemáticas. No cambies nada más de lo que te pedí.
```
Entrega: tus 8 preguntas con «k de N» por tipo, por qué falló la que falló y el único cambio que probarías después.